# Fifteen Ways to Predict Life Expectancy
**RGS Intro to Machine Learning** · Class demo

This notebook fits a wide range of prediction models — linear, regularized, additive, tree-based, instance-based, and neural — to the same task: predicting tract-level life expectancy from the USALEEP/SoVI dataset. Every model sees the same 67% train / 33% test split and the same 28 socioeconomic predictors, with **no cross-validation** (to keep 15+ model fits cheap) and a single fixed, "reasonable" hyperparameter setting per model — none of these are tuned.

The point isn't to crown a winner. It's to see how each model's results can be read, and **how each one tends to fail** — overfitting, instability, a scaling limit, or just the wrong shape of flexibility for this problem.

- **TODO:** none of these predictions come with an uncertainty interval. A later pass could add conformal prediction, quantile regression, or (for the GP) its native posterior variance.
- **Not addressed here:** these are census tracts, so nearby tracts have correlated life expectancy — a plain random holdout can leak information between train and test in a way a spatial holdout wouldn't. We covered that issue earlier in the course; this notebook ignores it to keep the comparison simple.


In [ ]:
from functools import reduce
import operator
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb
import lightgbm as lgb
from pygam import LinearGAM, s
from interpret.glassbox import ExplainableBoostingRegressor

RANDOM_STATE = 0


## Data

Tract-level SoVI variables merged with USALEEP life expectancy (`code/05_merge_data.py`). Predictors are the same 28 SoVI variables used in `PCA_demo.ipynb` — the geography columns (`geo_*`) are deliberately left out, since spatial handling is out of scope here (see note above).

Rows with any missing predictor or missing life expectancy are dropped outright (no imputation) — mostly zero-population tracts where percentages are undefined, plus tracts USALEEP didn't estimate.

In [ ]:
DATA = Path("data/processed")  # run from the repository root
tract = pd.read_csv(DATA / "sovi_tract_2015_merged.csv", dtype={"GEOID": str})

TARGET = "life_expectancy"
FEATURES = [c for c in tract.columns
            if c not in ("GEOID", "NAME", "population", TARGET) and not c.startswith("geo_")]
assert len(FEATURES) == 28

n_before = len(tract)
tract = tract.dropna(subset=FEATURES + [TARGET])
print(f"Dropped {n_before - len(tract):,} of {n_before:,} tracts with missing features or "
      f"life expectancy; {len(tract):,} remain.")
assert tract[FEATURES + [TARGET]].isna().sum().sum() == 0

tract[FEATURES + [TARGET]].describe().T[["mean", "std", "min", "max"]]


## Train/test split

A single random 67/33 holdout, reused by every model below — no cross-validation, no per-model tuning. (Tracts are spatially correlated, so this random split is a bit optimistic about generalization to genuinely new places; see the note above.)

In [ ]:
X = tract[FEATURES]
y = tract[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=RANDOM_STATE
)
print(f"{len(X_train):,} train / {len(X_test):,} test tracts")

# a smaller, shared test subsample reused below for permutation importance (repeated
# predictions get expensive on the full 21K-row test set, especially for GPR/KNN)
X_test_perm = X_test.sample(n=3_000, random_state=RANDOM_STATE)
y_test_perm = y_test.loc[X_test_perm.index]


In [ ]:
results = []


def evaluate(name, y_true, y_pred, fit_seconds=None):
    """Score one model's test predictions and append RMSE/MAE/R^2 to `results`."""
    row = {
        "model": name,
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred),
    }
    if fit_seconds is not None:
        row["fit_seconds"] = fit_seconds
    results.append(row)
    print(f"{name}: RMSE={row['rmse']:.2f}, MAE={row['mae']:.2f}, R2={row['r2']:.3f}")
    return row


def coefficient_table(term_names, raw_coefs, term_values, top_n=None):
    """Coefficient table sorted by |standardized coefficient| (coef * term's train-set SD),
    so terms on different scales (percent, dollars, a spline basis value, ...) are comparable.
    """
    term_stds = np.asarray(term_values).std(axis=0)
    table = pd.DataFrame({
        "term": term_names,
        "coefficient": raw_coefs,
        "standardized_coefficient": raw_coefs * term_stds,
    })
    table = table.reindex(table["standardized_coefficient"].abs().sort_values(ascending=False).index)
    table = table.reset_index(drop=True)
    return table.head(top_n) if top_n else table


def effect_curve(pipeline, feature, reference_row, grid):
    """Predicted life expectancy as `feature` varies over `grid`, every other feature held at
    `reference_row`, centered on the prediction at the reference itself so curves from
    different models/features start at 0 and are comparable on one y-scale.
    """
    X_grid = pd.DataFrame([reference_row.values] * len(grid), columns=reference_row.index)
    X_grid[feature] = grid
    baseline = pipeline.predict(reference_row.to_frame().T)[0]
    return pipeline.predict(X_grid) - baseline


def plot_sorted_bar(series, title, xlabel, diverging=False, color="#3b6fa0"):
    """Horizontal bar chart of an importance/coefficient series, largest at top."""
    series = series.sort_values()
    bar_colors = np.where(series.values >= 0, "#3b6fa0", "#b5542c") if diverging else color
    fig, ax = plt.subplots(figsize=(6, max(3, 0.35 * len(series))))
    ax.barh(series.index, series.values, color=bar_colors)
    if diverging:
        ax.axvline(0, color="#888888", linewidth=1)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


## 1. OLS (baseline)

Plain linear regression. Coefficients are easy to read — years of life expectancy per one-unit change in a predictor, holding the others fixed — which is exactly why it's the baseline every fancier model below should beat, and exactly the interpretability payoff a linear model offers: every predictor's effect is explicit, additive, and visible in one plot.

$$\hat{y} = \beta_0 + \sum_{j=1}^{28} \beta_j x_j, \qquad \hat\beta = \arg\min_\beta \|y - X\beta\|_2^2$$

The bar chart below uses coefficients from the *same* model fit on standardized predictors, purely so magnitudes are comparable across variables in different units (percent vs. dollars vs. years of age) — the predictions and the RMSE above come from the unscaled fit.

**Where this breaks:** the residual-vs-fitted plot shows visible curvature, meaning OLS is missing real nonlinearity in how these predictors relate to life expectancy. And several SoVI variables are strongly correlated with each other (e.g. poverty, income, and education), so individual coefficients — standardized or not — can be unstable even when the overall fit is fine: a coefficient's size and sign can shift noticeably if a correlated predictor is added or dropped.

In [ ]:
ols = LinearRegression().fit(X_train, y_train)
evaluate("OLS (baseline)", y_test, ols.predict(X_test))

ols_std_coefs = pd.Series(
    LinearRegression().fit(StandardScaler().fit_transform(X_train), y_train).coef_,
    index=FEATURES,
).sort_values()

fig, ax = plt.subplots(figsize=(6, 7))
ax.barh(
    ols_std_coefs.index, ols_std_coefs.values,
    color=np.where(ols_std_coefs.values >= 0, "#3b6fa0", "#b5542c"),
)
ax.axvline(0, color="#888888", linewidth=1)
ax.set_xlabel("Standardized OLS coefficient (years per 1 SD)")
ax.set_title("OLS coefficients — standardized for comparability")
plt.tight_layout()
plt.show()

fitted = ols.predict(X_train)
resid = y_train - fitted

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(fitted, resid, s=5, alpha=0.15, color="#3b6fa0")
ax.axhline(0, color="#888888", linewidth=1)
ax.set_xlabel("Fitted life expectancy (years)")
ax.set_ylabel("Residual (years)")
ax.set_title("OLS residuals vs. fitted — curvature hints at missing nonlinearity")
plt.tight_layout()
plt.show()


## 2. Linear regression + quadratic terms

Expand every predictor to include its square and all pairwise products, then fit OLS on the expanded set — deliberately *without* scaling first, to see what that costs.

$$\hat{y} = \beta_0 + \sum_j \beta_j x_j + \sum_j \gamma_j x_j^2 + \sum_{j<k} \delta_{jk}\, x_j x_k$$

Still exactly OLS — $\hat\beta = \arg\min_\beta \|y - X\beta\|_2^2$ — just fit on 434 expanded columns ($x_j$, $x_j^2$, $x_j x_k$) instead of the original 28.

**Where this breaks:** this isn't the overfitting demo it looks like it should be — train and test R² end up close to each other, but both are *worse* than plain OLS. Squaring and cross-multiplying 28 raw-scale predictors (percentages next to dollar amounts) produces ~400 columns on wildly different scales, which makes the unregularized least-squares fit poorly conditioned. Scaling first fixes this (train R² jumps from 0.52 to 0.62 — try it), which is exactly why the regularized models below scale before fitting. It also shows up directly in the effect plots further down: `per_capita_income`, the predictor with by far the largest raw numeric range, swamps every other predictor's panel on their shared scale.

In [ ]:
quad = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
quad.fit(X_train, y_train)

n_quad_features = quad.named_steps["polynomialfeatures"].transform(X_train.iloc[:1]).shape[1]
train_r2 = r2_score(y_train, quad.predict(X_train))
print(f"28 raw features -> {n_quad_features} after degree-2 expansion")
print(f"train R2={train_r2:.3f}")
evaluate("Linear regression + quadratic terms", y_test, quad.predict(X_test))

quad_scaled = make_pipeline(
    StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), LinearRegression()
)
quad_scaled.fit(X_train, y_train)
print(f"for comparison, scaling first gives train R2="
      f"{r2_score(y_train, quad_scaled.predict(X_train)):.3f}")


In [ ]:
quad_term_names = quad.named_steps["polynomialfeatures"].get_feature_names_out(FEATURES)
quad_terms_train = quad.named_steps["polynomialfeatures"].transform(X_train)
quad_table = coefficient_table(
    quad_term_names, quad.named_steps["linearregression"].coef_, quad_terms_train, top_n=20
)
print(f"showing top 20 of {len(quad_term_names)} terms, by |standardized coefficient|")
quad_table


## 3. Linear regression + splines

Instead of a single polynomial, give each predictor its own flexible spline curve (`SplineTransformer`, degree 3, 5 knots — fixed, not tuned), then fit OLS on the expanded basis.

$$\hat{y} = \beta_0 + \sum_{j=1}^{28} \sum_{m=1}^{M} \beta_{jm}\, B_m(x_j)$$

Each $x_j$ is replaced by $M$ basis functions $B_m$ (piecewise cubic pieces joined at the 5 knots) — still just OLS, now on $28 \times M$ columns instead of 28. The basis shape is fixed by the knots *before* fitting; only the $\beta_{jm}$ are estimated.

**Where this breaks:** splines let each predictor bend on its own, but the model is still additive — it can't represent one predictor's effect depending on another's value (e.g. poverty mattering differently in dense vs. rural tracts). The knot count we picked is also just a guess; a different choice could over- or under-fit the curve shape — the effect plots below show exactly where that shows up: a couple of predictors swing wildly in the sparsely-populated tails of their range, where a handful of outlier tracts give the spline very little to anchor on.

In [ ]:
splines = make_pipeline(SplineTransformer(degree=3, n_knots=5), LinearRegression())
splines.fit(X_train, y_train)
evaluate("Linear regression + splines", y_test, splines.predict(X_test))


In [ ]:
spline_term_names = splines.named_steps["splinetransformer"].get_feature_names_out(FEATURES)
spline_terms_train = splines.named_steps["splinetransformer"].transform(X_train)
spline_table = coefficient_table(
    spline_term_names, splines.named_steps["linearregression"].coef_, spline_terms_train, top_n=20
)
print(f"showing top 20 of {len(spline_term_names)} terms, by |standardized coefficient|")
spline_table


## Effect plots: quadratic vs. splines

Individual coefficients on expanded terms (`pct_asian^2`, `pct_asian pct_black`, `pct_asian_sp_3`, ...) aren't meaningful on their own — this is the real interpretability cost of both models above. What *is* interpretable is each predictor's **net effect**: hold every other predictor at its training-set median, vary one predictor across its observed range, and plot the resulting prediction (centered on the median tract, so every curve starts at 0).

Both models' panels below share one y-axis scale, so the height of a curve — not just its shape — is directly comparable across predictors *and* across the two models.

In [ ]:
reference = X_train[FEATURES].median()
grids = {f: np.linspace(X_train[f].min(), X_train[f].max(), 50) for f in FEATURES}

quad_curves = {f: effect_curve(quad, f, reference, grids[f]) for f in FEATURES}
spline_curves = {f: effect_curve(splines, f, reference, grids[f]) for f in FEATURES}

y_margin = 0.5
y_lo = min(c.min() for c in [*quad_curves.values(), *spline_curves.values()]) - y_margin
y_hi = max(c.max() for c in [*quad_curves.values(), *spline_curves.values()]) + y_margin


def plot_effect_grid(curves, title):
    """28-panel small-multiples grid of centered effect curves, sorted by swing, shared y-axis."""
    ordered = sorted(FEATURES, key=lambda f: curves[f].max() - curves[f].min(), reverse=True)
    fig, axes = plt.subplots(7, 4, figsize=(14, 18), sharey=True)
    for ax, feat in zip(axes.flat, ordered):
        ax.plot(grids[feat], curves[feat], color="#3b6fa0", linewidth=1.5)
        ax.axhline(0, color="#cccccc", linewidth=0.8)
        ax.set_ylim(y_lo, y_hi)
        ax.set_title(feat, fontsize=8)
        ax.tick_params(labelsize=6)
    fig.suptitle(title, y=1.0, fontsize=13)
    fig.supylabel("Predicted life expectancy vs. the median tract (years)")
    plt.tight_layout()
    plt.show()


plot_effect_grid(quad_curves, "Quadratic: effect of each predictor, others at their median")
plot_effect_grid(spline_curves, "Splines: effect of each predictor, others at their median")


## 4. LASSO

Scaling matters here: LASSO penalizes coefficient *magnitude*, so a predictor's raw units (e.g. percent vs. dollars) would otherwise distort which ones get penalized hardest. We standardize (fit on train only, inside the pipeline) before fitting.

$$\hat\beta = \arg\min_\beta \; \|y - X\beta\|_2^2 + \alpha \sum_j |\beta_j|$$

The $\sum_j |\beta_j|$ penalty (the $L_1$ norm of $\beta$) is what zeroes coefficients out completely, unlike Ridge's penalty below — it's also why this needs a numerical solver (coordinate descent) rather than a closed-form solution like OLS/Ridge have.

**Where this breaks:** `alpha` is fixed at a "reasonable" value, not chosen by cross-validation — the count of coefficients driven to exactly zero below is a direct consequence of that one arbitrary choice. A different fixed alpha could zero out everything, or nothing.

In [ ]:
lasso = make_pipeline(
    StandardScaler(), Lasso(alpha=0.1, max_iter=10_000, random_state=RANDOM_STATE)
)
lasso.fit(X_train, y_train)
evaluate("LASSO", y_test, lasso.predict(X_test))

coefs = lasso.named_steps["lasso"].coef_
print(f"{(coefs == 0).sum()} of {len(coefs)} coefficients zeroed out at alpha=0.1")


In [ ]:
# lasso.coef_ is already standardized (fit on scaled features); divide by each
# feature's raw-scale SD to recover a "years per one natural unit" coefficient too.
lasso_table = coefficient_table(FEATURES, coefs / X_train[FEATURES].std().values, X_train[FEATURES].values)
plot_sorted_bar(
    lasso_table.set_index("term")["standardized_coefficient"],
    "LASSO: standardized coefficients", "Standardized coefficient (years per 1 SD)",
    diverging=True,
)
lasso_table


## 5. Ridge regression

Same scaling reasoning as LASSO, but Ridge penalizes squared coefficient magnitude instead of absolute magnitude.

$$\hat\beta = \arg\min_\beta \; \|y - X\beta\|_2^2 + \alpha \sum_j \beta_j^2$$

Squaring the penalty ($L_2$ norm) keeps it differentiable everywhere — including at $\beta_j = 0$ — which is exactly why it shrinks coefficients smoothly toward zero but, unlike LASSO's $L_1$ penalty, never snaps one to exactly zero.

**Where this breaks:** Ridge shrinks every coefficient toward zero but — unlike LASSO — never zeroes one out completely, so it does no feature selection. With this dataset's collinear SoVI variables, that means credit for an effect gets spread across several correlated predictors in a way that's hard to attribute to any one of them.

In [ ]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1.0, random_state=RANDOM_STATE))
ridge.fit(X_train, y_train)
evaluate("Ridge", y_test, ridge.predict(X_test))


In [ ]:
ridge_coefs = ridge.named_steps["ridge"].coef_
ridge_table = coefficient_table(FEATURES, ridge_coefs / X_train[FEATURES].std().values, X_train[FEATURES].values)
plot_sorted_bar(
    ridge_table.set_index("term")["standardized_coefficient"],
    "Ridge: standardized coefficients", "Standardized coefficient (years per 1 SD)",
    diverging=True,
)
ridge_table


## 6. Elastic Net

A 50/50 mix of the LASSO and Ridge penalties (`l1_ratio=0.5`), so it sits between the two: some coefficients zeroed, the rest shrunk.

$$\hat\beta = \arg\min_\beta \; \|y - X\beta\|_2^2 + \alpha \left(\rho \sum_j |\beta_j| + \frac{1-\rho}{2} \sum_j \beta_j^2\right), \quad \rho = 0.5$$

$\rho$ (`l1_ratio`) just dials between the two penalties above: $\rho=1$ is plain LASSO, $\rho=0$ is plain Ridge. One more untuned knob stacked on top of `alpha`.

In [ ]:
enet = make_pipeline(
    StandardScaler(),
    ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10_000, random_state=RANDOM_STATE),
)
enet.fit(X_train, y_train)
evaluate("Elastic Net", y_test, enet.predict(X_test))

coefs = enet.named_steps["elasticnet"].coef_
print(f"{(coefs == 0).sum()} of {len(coefs)} coefficients zeroed out")


In [ ]:
enet_table = coefficient_table(FEATURES, coefs / X_train[FEATURES].std().values, X_train[FEATURES].values)
plot_sorted_bar(
    enet_table.set_index("term")["standardized_coefficient"],
    "Elastic Net: standardized coefficients", "Standardized coefficient (years per 1 SD)",
    diverging=True,
)
enet_table


## 7. Generalized Additive Model

Like the spline model above, but fit with `pygam`'s own smoothing penalty per term instead of plugging a spline basis into plain OLS.

$$\hat{y} = \beta_0 + \sum_{j=1}^{28} f_j(x_j), \qquad \hat{f} = \arg\min_f \; \|y - \hat y\|_2^2 + \sum_j \lambda_j \int f_j''(x)^2\, dx$$

Same additive form as the splines model, but now each $f_j$ is penalized directly for *wiggliness* (that integral of its second derivative) instead of being left to plain, unpenalized OLS on a fixed basis — a smoothness knob the splines model above doesn't have.

**Where this breaks:** fitting time is already noticeably longer than any linear model, for a model that's still additive — no interactions between predictors, same limitation as the spline regression above.

In [ ]:
gam_start = time.time()
terms = reduce(operator.add, (s(i) for i in range(len(FEATURES))))
gam = LinearGAM(terms)
gam.fit(X_train.values, y_train.values)
gam_seconds = time.time() - gam_start

evaluate("GAM", y_test, gam.predict(X_test.values), fit_seconds=gam_seconds)
print(f"fit in {gam_seconds:.1f}s")


`pygam` has its own built-in partial-dependence plotting (`generate_X_grid` + `partial_dependence`), which also comes with a 95% confidence band for free — the one place in this notebook with any native uncertainty on an individual effect (see the TODO at the top). Panels are ordered by effect size and share one y-axis, same as the quadratic/spline comparison above.

In [ ]:
gam_grids, gam_effects, gam_bands = {}, {}, {}
for i, feat in enumerate(FEATURES):
    XX = gam.generate_X_grid(term=i)
    gam_grids[feat] = XX[:, i]
    gam_effects[feat] = gam.partial_dependence(term=i, X=XX)
    gam_bands[feat] = gam.partial_dependence(term=i, X=XX, width=0.95)[1]

gam_y_lo = min(b[:, 0].min() for b in gam_bands.values())
gam_y_hi = max(b[:, 1].max() for b in gam_bands.values())
gam_order = sorted(FEATURES, key=lambda f: gam_effects[f].max() - gam_effects[f].min(), reverse=True)

fig, axes = plt.subplots(7, 4, figsize=(14, 18), sharey=True)
for ax, feat in zip(axes.flat, gam_order):
    ax.plot(gam_grids[feat], gam_effects[feat], color="#3b6fa0", linewidth=1.5)
    ax.fill_between(
        gam_grids[feat], gam_bands[feat][:, 0], gam_bands[feat][:, 1], color="#3b6fa0", alpha=0.2
    )
    ax.axhline(0, color="#cccccc", linewidth=0.8)
    ax.set_ylim(gam_y_lo, gam_y_hi)
    ax.set_title(feat, fontsize=8)
    ax.tick_params(labelsize=6)
fig.suptitle("GAM: partial dependence per predictor (95% CI shaded)", y=1.0, fontsize=13)
fig.supylabel("Partial effect on life expectancy (years)")
plt.tight_layout()
plt.show()


## 8. Explainable Boosting Machine

A boosted additive model like the GAM above, but EBMs also screen all pairwise combinations of predictors and keep the `interactions=10` most useful as explicit interaction terms — so, unlike every other model in this section, it isn't purely additive.

$$\hat{y} = \beta_0 + \sum_{j=1}^{28} f_j(x_j) + \sum_{(j,k) \in \text{top }10} f_{jk}(x_j, x_k)$$

Each $f_j$ (and $f_{jk}$) is itself a sum of shallow decision-tree stumps, fit one at a time by cycling through every term and boosting it a little — same additive structure as the GAM's $f_j$ above, just built from trees instead of smoothing splines, and with the $f_{jk}$ terms giving it the one thing the GAM structurally can't have.

**Where this breaks:** fitting time is the longest yet. And "10 interactions" is itself an untuned, arbitrary choice — a different number could surface different pairs, or miss a real one entirely; nothing here confirms 10 is the right count.

In [ ]:
ebm_start = time.time()
ebm = ExplainableBoostingRegressor(interactions=10, random_state=RANDOM_STATE)
ebm.fit(X_train, y_train)
ebm_seconds = time.time() - ebm_start

evaluate("EBM", y_test, ebm.predict(X_test), fit_seconds=ebm_seconds)
print(f"fit in {ebm_seconds:.1f}s")

importance = (
    pd.Series(ebm.term_importances(), index=ebm.term_names_)
    .sort_values(ascending=True)
    .tail(10)
)
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(importance.index, importance.values, color="#3b6fa0")
ax.set_xlabel("EBM term importance")
ax.set_title("EBM: 10 most important terms (main effects + interactions)")
plt.tight_layout()
plt.show()


EBM has its own built-in shape functions, one per term — same idea as the GAM's partial dependence above, and plotted the same way: panels ordered by effect size, one shared y-axis, 95% CI shaded, so this grid is directly comparable to the GAM, quadratic, and spline grids elsewhere in the notebook.

In [ ]:
from IPython.display import display

ebm_explanation = ebm.explain_global()

ebm_grids, ebm_effects, ebm_bands = {}, {}, {}
for i, feat in enumerate(FEATURES):
    d = ebm_explanation.data(i)
    ebm_grids[feat] = np.asarray(d["names"])
    ebm_effects[feat] = np.append(d["scores"], d["scores"][-1])
    ebm_bands[feat] = (
        np.append(d["lower_bounds"], d["lower_bounds"][-1]),
        np.append(d["upper_bounds"], d["upper_bounds"][-1]),
    )

ebm_y_lo = min(b[0].min() for b in ebm_bands.values())
ebm_y_hi = max(b[1].max() for b in ebm_bands.values())
ebm_order = sorted(FEATURES, key=lambda f: ebm_effects[f].max() - ebm_effects[f].min(), reverse=True)

fig, axes = plt.subplots(7, 4, figsize=(14, 18), sharey=True)
for ax, feat in zip(axes.flat, ebm_order):
    ax.step(ebm_grids[feat], ebm_effects[feat], where="post", color="#3b6fa0", linewidth=1.5)
    ax.fill_between(
        ebm_grids[feat], ebm_bands[feat][0], ebm_bands[feat][1],
        step="post", color="#3b6fa0", alpha=0.2,
    )
    ax.axhline(0, color="#cccccc", linewidth=0.8)
    ax.set_ylim(ebm_y_lo, ebm_y_hi)
    ax.set_title(feat, fontsize=8)
    ax.tick_params(labelsize=6)
fig.suptitle("EBM: shape function per predictor (95% CI shaded)", y=1.0, fontsize=13)
fig.supylabel("Contribution to predicted life expectancy (years)")
plt.tight_layout()
plt.show()


### EBM's most important interactions

The 4 pairwise terms with the highest importance among the 10 EBM considered, as built-in heatmaps (`interpret` fixes the color scale to the model's own score range by default, so these are already comparable to each other and to the shape functions above — darker red/blue means a bigger positive/negative swing). A diagonal or blocky pattern here is the signature of a real interaction: one predictor's effect changing depending on the other's value, which none of the additive models elsewhere in this notebook (GAM, splines, LASSO, ...) can represent at all.

In [ ]:
interaction_idx = [i for i, feats in enumerate(ebm.term_features_) if len(feats) == 2]
top_interactions = sorted(interaction_idx, key=lambda i: ebm.term_importances()[i], reverse=True)[:4]

interaction_importance = pd.Series(
    {ebm.term_names_[i]: ebm.term_importances()[i] for i in top_interactions}
).sort_values()
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(interaction_importance.index, interaction_importance.values, color="#b5542c")
ax.set_xlabel("EBM term importance")
ax.set_title("EBM: most important interaction terms")
plt.tight_layout()
plt.show()

for i in top_interactions:
    display(ebm_explanation.visualize(int(i)))


## Comparing functional forms across the additive models

All four additive-ish models above (quadratic, splines, GAM, EBM main effects) estimate a separate curve for each predictor — here they're overlaid, one panel per predictor, so you can see directly whether they agree on the *shape* of a predictor's relationship with life expectancy, or whether that shape is a model-specific artifact.

A few things to flag about how these curves were built, since they weren't all built the same way. Quadratic and splines are already centered at the prediction for the median tract. EBM turns out to center itself at nearly the same point by its own convention (checked numerically). GAM does **not** — `pygam` doesn't enforce each term to average to zero, so individual terms can silently absorb part of the intercept (one term here carried a mean of +1.3 years with nothing forcing it toward 0). Left uncorrected, GAM's curves would sit offset from the other three for no substantive reason, so every curve below is explicitly re-centered to read 0 at that predictor's own training-median value before plotting — **don't read the exact zero-crossing as independently meaningful, but it's now on equal footing across all four models**, so shape and relative amplitude are directly comparable. And unlike the earlier grids, **each panel here has its own y-axis**: the comparison that matters now is the four models against each other for one predictor at a time, not one predictor against another, and letting each panel scale to its own data keeps small-amplitude predictors readable even on the couple of panels (`per_capita_income` especially) where the unregularized quadratic model swings far more than the other three. Panels are ordered by EBM's own importance ranking.

In [ ]:
from matplotlib.lines import Line2D


def center_at_median(grid, effect, feat):
    """Shift a curve to read 0 at `feat`'s own training-median, so models with different
    native centering conventions (quadratic/splines/EBM vs. GAM) are directly comparable.
    """
    idx = min(np.searchsorted(grid, X_train[feat].median()), len(effect) - 1)
    return effect - effect[idx]


model_styles = {
    "Quadratic": ("#3b6fa0", "-"),
    "Splines": ("#e08214", "--"),
    "GAM": ("#4daf4a", ":"),
    "EBM": ("#984ea3", "-."),
}
form_order = sorted(FEATURES, key=lambda f: ebm.term_importances()[FEATURES.index(f)], reverse=True)

fig, axes = plt.subplots(7, 4, figsize=(16, 20))
for ax, feat in zip(axes.flat, form_order):
    ax.plot(grids[feat], center_at_median(grids[feat], quad_curves[feat], feat), color=model_styles["Quadratic"][0], linestyle=model_styles["Quadratic"][1], linewidth=1.5)
    ax.plot(grids[feat], center_at_median(grids[feat], spline_curves[feat], feat), color=model_styles["Splines"][0], linestyle=model_styles["Splines"][1], linewidth=1.5)
    ax.plot(gam_grids[feat], center_at_median(gam_grids[feat], gam_effects[feat], feat), color=model_styles["GAM"][0], linestyle=model_styles["GAM"][1], linewidth=1.5)
    ax.step(ebm_grids[feat], center_at_median(ebm_grids[feat], ebm_effects[feat], feat), where="post", color=model_styles["EBM"][0], linestyle=model_styles["EBM"][1], linewidth=1.5)
    ax.axhline(0, color="#cccccc", linewidth=0.8)
    ax.set_title(feat, fontsize=8)
    ax.tick_params(labelsize=6)

handles = [Line2D([0], [0], color=c, linestyle=ls, label=name) for name, (c, ls) in model_styles.items()]
fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 1.0))
fig.suptitle("Functional form by model, ordered by EBM importance", y=0.975, fontsize=14)
fig.supylabel("Effect on predicted life expectancy (years)")
plt.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()


## 9. CART (classification and regression trees)

Two fits on purpose: an **unconstrained** tree (grows until every leaf is pure or tiny) and a **shallow** one (`max_depth=5`, `min_samples_leaf=50`).

**Where this breaks:** this is the clearest overfitting demo in the notebook. The unconstrained tree gets a near-perfect train R² by memorizing individual tracts — compare its train R² and depth to the shallow tree's, and compare both trees' test RMSE to see which one actually generalizes.

In [ ]:
cart_deep = DecisionTreeRegressor(random_state=RANDOM_STATE)
cart_deep.fit(X_train, y_train)
print(f"unconstrained tree depth={cart_deep.get_depth()}, "
      f"train R2={r2_score(y_train, cart_deep.predict(X_train)):.3f}")
evaluate("CART (unconstrained)", y_test, cart_deep.predict(X_test))

cart_shallow = DecisionTreeRegressor(
    max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
)
cart_shallow.fit(X_train, y_train)
print(f"shallow tree train R2={r2_score(y_train, cart_shallow.predict(X_train)):.3f}")
evaluate("CART (max_depth=5)", y_test, cart_shallow.predict(X_test))


Feature importance (Gini: how much a feature reduces impurity, averaged over every split that uses it) is the typical way to rank predictors for a tree — below, the unconstrained and shallow trees' rankings side by side. And since the shallow tree is only 5 levels deep, we can draw it — truncated to the top 3 levels below for legibility (32 leaves at depth 5 is too many to render as readable text), which is still more of the decision logic laid bare than any other model in this notebook shows.

In [ ]:
cart_top = pd.Series(cart_shallow.feature_importances_, index=FEATURES).sort_values(ascending=False).head(10).index
y_pos = np.arange(len(cart_top))

fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(
    y_pos - 0.2, pd.Series(cart_deep.feature_importances_, index=FEATURES)[cart_top],
    height=0.4, color="#b5542c", label="unconstrained",
)
ax.barh(
    y_pos + 0.2, pd.Series(cart_shallow.feature_importances_, index=FEATURES)[cart_top],
    height=0.4, color="#3b6fa0", label="max_depth=5",
)
ax.set_yticks(y_pos, cart_top)
ax.invert_yaxis()
ax.set_xlabel("Gini importance")
ax.set_title("CART: feature importance, unconstrained vs. shallow")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(20, 8), dpi=120)
plot_tree(
    cart_shallow, max_depth=3, feature_names=FEATURES, filled=True,
    fontsize=8, precision=0, ax=ax, impurity=False,
)
ax.set_title("CART (max_depth=5): top 3 of 5 levels")
plt.tight_layout()
plt.show()


## 10. Random Forest

200 trees, each capped at `max_depth=10`, averaged together.

**Where this breaks (less than CART did):** averaging many trees reduces the variance that sank the unconstrained CART above — compare the train/test gap here to CART's. The cost is losing the single-tree's readability; "the model" is now 200 trees, not one diagram.

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
print(f"train R2={r2_score(y_train, rf.predict(X_train)):.3f}")
evaluate("Random Forest", y_test, rf.predict(X_test))


Two different "importance" measures, deliberately shown side by side. **Impurity importance** (`.feature_importances_`, same Gini measure as CART above) is built in and free, but it's a known-biased estimate — it's inflated for continuous predictors with many unique values (more candidate split points) relative to coarser ones, regardless of whether they're actually more predictive. **Permutation importance** avoids that bias by measuring something model-agnostic instead: how much test R² drops when a predictor's values are shuffled. Where the two rankings disagree is exactly where impurity importance was probably misleading you.

In [ ]:
rf_impurity = pd.Series(rf.feature_importances_, index=FEATURES)
rf_perm = permutation_importance(
    rf, X_test_perm, y_test_perm, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
rf_permutation = pd.Series(rf_perm.importances_mean, index=FEATURES)

plot_sorted_bar(rf_impurity.sort_values().tail(10), "Random Forest: impurity importance", "Gini importance")
plot_sorted_bar(
    rf_permutation.sort_values().tail(10), "Random Forest: permutation importance",
    "Mean decrease in test R2 when shuffled",
)


## 11. XGBoost

500 boosting rounds, shallow trees (`max_depth=4`), a small learning rate — and, deliberately, **no early stopping or held-out validation set**.

**Where this breaks:** with nothing to stop it, boosting keeps reducing training error for all 500 rounds regardless of whether that's still helping the test set — check the train/test R² gap.

In [ ]:
xgbr = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4,
    random_state=RANDOM_STATE, n_jobs=-1,
)
xgbr.fit(X_train, y_train)
print(f"train R2={r2_score(y_train, xgbr.predict(X_train)):.3f}")
evaluate("XGBoost", y_test, xgbr.predict(X_test))

plot_sorted_bar(
    pd.Series(xgbr.feature_importances_, index=FEATURES).sort_values().tail(10),
    "XGBoost: feature importance (gain)", "Normalized gain",
)


## 12. LightGBM

Same setup as XGBoost above (same rounds, depth, learning rate, no early stopping) — a direct comparison between two gradient boosting implementations on identical hyperparameters.

In [ ]:
lgbr = lgb.LGBMRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=4, importance_type="gain",
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
lgbr.fit(X_train, y_train)
print(f"train R2={r2_score(y_train, lgbr.predict(X_train)):.3f}")
evaluate("LightGBM", y_test, lgbr.predict(X_test))

plot_sorted_bar(
    pd.Series(lgbr.feature_importances_, index=FEATURES).sort_values().tail(10),
    "LightGBM: feature importance (gain)", "Total gain",
)


## 13. K-nearest neighbors

Predict a tract's life expectancy as the average of its `k=15` nearest neighbors in (scaled) feature space.

**Where this breaks:** scaling fixes the units problem, but not the dimensionality one — in 28 dimensions, a tract's "15 nearest neighbors" are often not actually close. Compare KNN's test RMSE to the simpler linear models above to see the curse of dimensionality in practice.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15))
knn.fit(X_train, y_train)
evaluate("KNN (k=15)", y_test, knn.predict(X_test))


KNN has no coefficients and no splits to read off — there's nothing built in to ask "which predictor mattered?" The standard model-agnostic fallback, used for the rest of the black-box models below too, is **permutation importance**: shuffle one predictor's values across a test sample (breaking its link to the outcome) and measure how much worse predictions get; a bigger drop means that predictor was doing more work. Computed on the 3,000-row test subsample from earlier to keep this fast — a smaller sample makes the ranking noisier, which is itself part of the cost of not having a native importance measure.

In [ ]:
knn_perm = permutation_importance(
    knn, X_test_perm, y_test_perm, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
plot_sorted_bar(
    pd.Series(knn_perm.importances_mean, index=FEATURES).sort_values().tail(10),
    "KNN: permutation importance", "Mean decrease in test R2 when shuffled",
)


## 14. Gaussian Process Regression

GPR training cost scales as O(n³) in the number of training rows. Fitting on the full training split (tens of thousands of tracts) isn't practical here, so this fit uses a random 2,000-row **subsample** of the training data — everything else still predicts and gets scored on the full test set.

**Where this breaks:** the subsampling requirement itself *is* the failure mode — not a bad kernel choice. Throwing away over 95% of the training data to make GPR tractable is a real cost, and it shows up as badly worse performance than every other model, several of which were fit on the full training set in a fraction of the time. (You may also see a convergence warning about the kernel's length-scale hyperparameter hitting its bound — a small subsample and untuned bounds don't give the optimizer much to work with.)

In [ ]:
GP_SAMPLE_SIZE = 2_000
gp_sample = X_train.sample(n=GP_SAMPLE_SIZE, random_state=RANDOM_STATE).index
X_train_gp, y_train_gp = X_train.loc[gp_sample], y_train.loc[gp_sample]

gp_start = time.time()
gp = make_pipeline(
    StandardScaler(),
    GaussianProcessRegressor(kernel=RBF() + WhiteKernel(), random_state=RANDOM_STATE),
)
gp.fit(X_train_gp, y_train_gp)
gp_seconds = time.time() - gp_start

evaluate(
    "Gaussian Process (2K-row subsample)", y_test, gp.predict(X_test), fit_seconds=gp_seconds
)
print(f"fit on {GP_SAMPLE_SIZE:,} of {len(X_train):,} training rows in {gp_seconds:.1f}s")


Same model-agnostic approach as KNN above, on the same 3,000-row test subsample (GPR prediction cost scales with how many training points it has to compare against, so this is already about as fast as this model gets here).

In [ ]:
gp_perm = permutation_importance(
    gp, X_test_perm, y_test_perm, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
plot_sorted_bar(
    pd.Series(gp_perm.importances_mean, index=FEATURES).sort_values().tail(10),
    "Gaussian Process: permutation importance", "Mean decrease in test R2 when shuffled",
)


## 15. Neural net (MLP)

A small feed-forward network (two hidden layers, 32 then 16 units), with early stopping on an internal validation split and — like every other model here — no architecture tuning. Three identical fits, differing only in random weight initialization (`seed`).

**Where this breaks:** not the way you might expect. The test RMSE barely moves across seeds here — early stopping and 44K training rows are enough to make this particular architecture fairly reproducible (that's not guaranteed in general; turn off `early_stopping` or shrink the training set and it reappears). The real failure mode shows up in the leaderboard below instead: this is the most opaque, slowest-to-fit, hardest-to-explain model in the whole notebook, and it does not clearly beat the GAM, EBM, or Random Forest above — all of which are easier to interpret and were also fit with no tuning.

In [ ]:
mlp_seed_results, mlp_models = [], {}
for seed in (0, 1, 2):
    mlp = make_pipeline(
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32, 16), early_stopping=True,
            max_iter=1000, random_state=seed,
        ),
    )
    mlp.fit(X_train, y_train)
    mlp_models[seed] = mlp
    mlp_seed_results.append(
        evaluate(f"Neural net (MLP, seed={seed})", y_test, mlp.predict(X_test))
    )

rmse_spread = max(r["rmse"] for r in mlp_seed_results) - min(r["rmse"] for r in mlp_seed_results)
print(f"test RMSE spread across 3 random seeds alone: {rmse_spread:.2f} years")


One more opaque model, one more permutation-importance fallback — computed on the seed=0 fit only, on the same 3,000-row subsample as before. The point here is just to see whether the MLP's implicit feature usage lines up with the simpler, native-coefficient models above, not to re-litigate seed variance.

In [ ]:
mlp_perm = permutation_importance(
    mlp_models[0], X_test_perm, y_test_perm, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1
)
plot_sorted_bar(
    pd.Series(mlp_perm.importances_mean, index=FEATURES).sort_values().tail(10),
    "Neural net (MLP, seed=0): permutation importance", "Mean decrease in test R2 when shuffled",
)


## Leaderboard

Every model above, ranked by test RMSE. A few models appear more than once on purpose (the two CART depths, the three MLP seeds) — those duplicates are part of the failure-mode point, not an accident.

Remember the two caveats from the top: no prediction here carries an uncertainty interval (TODO), and the random holdout doesn't account for spatial correlation between nearby tracts.

In [ ]:
leaderboard = pd.DataFrame(results).sort_values("rmse").reset_index(drop=True)
leaderboard


In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
is_baseline = leaderboard["model"].eq("OLS (baseline)")
colors = np.where(is_baseline, "#9a9a9a", "#3b6fa0")

ax.barh(leaderboard["model"], leaderboard["rmse"], color=colors)
ax.invert_yaxis()  # best (lowest RMSE) at top
ax.set_xlabel("Test RMSE (years)")
ax.set_title("Model comparison on the 33% holdout")
for i, v in enumerate(leaderboard["rmse"]):
    ax.text(v + 0.02, i, f"{v:.2f}", va="center", fontsize=8, color="#444444")

from matplotlib.patches import Patch
ax.legend(
    handles=[Patch(color="#9a9a9a", label="OLS (baseline)"),
             Patch(color="#3b6fa0", label="Other models")],
    loc="upper right", frameon=False,
)
plt.tight_layout()
plt.show()
